# **Day 85: Pandas - Merge, Pivot Tables and Time Series**

In [1]:
import pandas as pd
import numpy as np

## **Merge: Joining Two Tables**
**Merge** joins two tables using a **common column**, like VLOOKUP in Excel or a JOIN in SQL.

In GIS, this is called a **table join**: for example, joining census data to a district map using the district name.

In [2]:
students = pd.DataFrame({
    "roll": [1, 2, 3, 4, 5],
    "name": ["Arpita", "Bikash", "Chaitali", "Dibyendu", "Esha"],
    "school_id": [10, 20, 10, 30, 99],
})
schools = pd.DataFrame({
    "school_id": [10, 20, 30, 40],
    "school": ["Midnapore Collegiate", "Hindu School", "Siliguri Boys' High", "Darjeeling Govt."],
    "district": ["Paschim Medinipur", "Kolkata", "Darjeeling", "Darjeeling"],
})
pd.merge(students, schools, on="school_id")                  # Inner join: only matching rows

,roll,name,school_id,school,district
0,1,Arpita,10,Midnapore Collegiate,Paschim Medinipur
1,2,Bikash,20,Hindu School,Kolkata
2,3,Chaitali,10,Midnapore Collegiate,Paschim Medinipur
3,4,Dibyendu,30,Siliguri Boys' High,Darjeeling


In [3]:
pd.merge(students, schools, on="school_id", how="left")      # Left join: keep all students

,roll,name,school_id,school,district
0,1,Arpita,10,Midnapore Collegiate,Paschim Medinipur
1,2,Bikash,20,Hindu School,Kolkata
2,3,Chaitali,10,Midnapore Collegiate,Paschim Medinipur
3,4,Dibyendu,30,Siliguri Boys' High,Darjeeling
4,5,Esha,99,NaN,NaN


| Join type | Keeps |
|---|---|
| `inner` | Only rows found in **both** tables |
| `left` | **All** rows of the left table |
| `right` | **All** rows of the right table |
| `outer` | **All** rows of both tables |

Esha's school ID (99) does not exist in the schools table, so her school is empty (NaN) in the left join.

## **Pivot Table**
A pivot table **summarises** data by putting one column's values as **rows** and another's as **columns**, just like Excel's PivotTable.

In [4]:
rng = np.random.default_rng(3)
sales = pd.DataFrame({
    "month": np.repeat(["Sep", "Oct", "Nov"], 6),
    "branch": np.tile(["Kolkata", "Kolkata", "Siliguri", "Siliguri", "Midnapore", "Midnapore"], 3),
    "item": np.tile(["Rosogolla", "Sandesh"], 9),
    "revenue": rng.integers(8000, 40000, 18),
})
pivot = sales.pivot_table(index="branch", columns="month", values="revenue", aggfunc="sum", margins=True)
pivot[["Sep", "Oct", "Nov", "All"]]

month,Sep,Oct,Nov,All
branch,,,,
Kolkata,44708,62444,51208,158360
Midnapore,47443,40490,61631,149564
Siliguri,29319,20272,29584,79175
All,121470,123206,142423,387099


## **melt: From Wide to Long**
`melt` is the opposite of pivot.

In [5]:
wide = pd.DataFrame({"city": ["Kolkata", "Darjeeling"], "Jan": [20, 6], "May": [31, 16], "Jul": [30, 18]})
long = wide.melt(id_vars="city", var_name="month", value_name="temp")
long

,city,month,temp
0,Kolkata,Jan,20
1,Darjeeling,Jan,6
2,Kolkata,May,31
3,Darjeeling,May,16
4,Kolkata,Jul,30
5,Darjeeling,Jul,18


## **Time Series**
A **time series** is data recorded over time, like daily temperature or monthly NDVI. Pandas is excellent for time series, when the **dates are used as the index**.

## **Example from Remote Sensing: NDVI Time Series of a Paddy Field**
A satellite passes over the same field every few days, so we can follow how the **crop grows** through the year. This is called **crop monitoring** or **phenology**.

In West Bengal, **Aman paddy** (the main rice crop) is planted in the monsoon (June-July) and harvested in November-December. So the field's NDVI should:
- be **low** before planting (bare, wet soil)
- **rise** as the rice grows (peak around September-October)
- **fall** after the harvest

Below is a **simulated** NDVI series for a paddy field in Paschim Medinipur, every 5 days (like Sentinel-2), with some cloudy dates missing.

In [6]:
dates = pd.date_range("2024-01-01", "2024-12-31", freq="5D")
day_of_year = dates.dayofyear.to_numpy()

# Aman rice growth curve: rises from July, peaks in October, falls after harvest
growth = np.exp(-((day_of_year - 280) ** 2) / (2 * 35 ** 2))
ndvi_values = 0.18 + 0.62 * growth + rng.normal(0, 0.03, len(dates))

ndvi = pd.Series(ndvi_values.round(3), index=dates, name="ndvi")

# Monsoon clouds: many dates in June to September have no valid data
cloudy = (dates.month.isin([6, 7, 8, 9])) & (rng.random(len(dates)) < 0.6)
ndvi[cloudy] = np.nan
print("Total dates:", len(ndvi), "| Cloudy (missing):", ndvi.isna().sum())
ndvi.head()

Total dates: 74 | Cloudy (missing): 15


2024-01-01    0.280
2024-01-06    0.187
2024-01-11    0.169
2024-01-16    0.172
2024-01-21    0.160
Freq: 5D, Name: ndvi, dtype: float64

## **Resample: Monthly Values**
`resample("ME")` groups a time series by **month** (ME = month end).

In [7]:
monthly = ndvi.resample("ME").mean().round(3)
monthly.index = monthly.index.strftime("%b")
monthly

Jan    0.183
Feb    0.186
Mar    0.195
Apr    0.185
May    0.162
Jun    0.178
Jul    0.233
Aug    0.362
Sep    0.635
Oct    0.747
Nov    0.518
Dec    0.280
Name: ndvi, dtype: float64

## **Filling Gaps: Interpolation**
For cloudy dates, we can estimate the missing NDVI from the dates before and after. This is called **interpolation**.

In [8]:
filled = ndvi.interpolate(method="time")
print("Missing before:", ndvi.isna().sum(), "| after:", filled.isna().sum())

peak_date = filled.idxmax()
print(f"Peak NDVI {filled.max():.2f} on {peak_date:%d %B %Y}")

Missing before: 15 | after: 0
Peak NDVI 0.81 on 12 October 2024


## **Rolling Average: Smoothing the Noise**

In [9]:
smooth = filled.rolling(window=5, center=True).mean()
pd.DataFrame({"raw": ndvi, "filled": filled.round(3), "smooth": smooth.round(3)})["2024-09-01":"2024-10-31"]

,raw,filled,smooth
2024-09-02,NaN,0.580,0.573
2024-09-07,0.635,0.635,0.614
2024-09-12,NaN,0.655,0.648
2024-09-17,NaN,0.675,0.675
2024-09-22,NaN,0.695,0.695
2024-09-27,NaN,0.715,0.718
2024-10-02,0.735,0.735,0.746
2024-10-07,0.772,0.772,0.768
2024-10-12,0.812,0.812,0.760
2024-10-17,0.805,0.805,0.749


## **Finding the Growing Season**
Let us say the crop is "growing" when the smoothed NDVI is above **0.4**.

In [10]:
growing = smooth[smooth > 0.4]
print("Growing season starts:", growing.index.min().strftime("%d %B"))
print("Growing season ends  :", growing.index.max().strftime("%d %B"))
print("Length:", (growing.index.max() - growing.index.min()).days, "days")

Growing season starts: 18 August
Growing season ends  : 21 November
Length: 95 days


This matches the Aman rice calendar well: the crop is green from around August to November.

In [11]:
# Quick question
# What is the difference between resample("ME").mean() and rolling(5).mean()?